# GigaGraph 3.2B: SOTA-Grade LLM Pre-Training
### Distributed Cold Start | Remote Source (GitHub Sync)

**Hardware Target:** Kaggle Dual T4 (2x16GB)
**Architecture:** GigaGraph v8.3.2 (Signal Stabilized)
**Features:** Two-Slot Checkpointing, Variance Scaling, Lower LR Gradient Descent


In [1]:
# 1. Environment & Auth
import os, sys
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = os.environ['HOME'] + '/.local/bin:' + os.environ['PATH']

from dotenv import load_dotenv
from kaggle_secrets import UserSecretsClient
load_dotenv()

try:
    sc = UserSecretsClient()
    hf_token = sc.get_secret('HF_TOKEN')
    wandb_key = sc.get_secret('WANDB_API_KEY')
except:
    hf_token, wandb_key = os.getenv('HF_TOKEN'), os.getenv('WANDB_API_KEY')

import torch, wandb
from huggingface_hub import login
if hf_token: login(token=hf_token)
if wandb_key: wandb.login(key=wandb_key)


In [2]:
# 2. Remote Synchronize (GitHub -> Kaggle)
import importlib, os, sys
REPO_URL = 'https://github.com/ey3lock3r/gnn-llm.git'

print('Synchronizing source from ' + REPO_URL + '...')
if not os.path.exists('.git'):
    !git init .
!git remote add origin {REPO_URL} || git remote set-url origin {REPO_URL}
!git fetch origin
!git reset --hard origin/master

!uv sync
if os.getcwd() not in sys.path: sys.path.append(os.getcwd())

import aptp_gnn, data_pipeline
importlib.reload(aptp_gnn)
importlib.reload(data_pipeline)

from aptp_gnn import GigaGraph_3B
from data_pipeline import GigaDataPipeline
from tqdm import tqdm


In [3]:
# 3. GigaGraph 3.2B Stabilized Training Loop
VOCAB_SIZE, D_MODEL, DEPTH = 128256, 3072, 32
LEARNING_RATE, SAVE_INTERVAL = 1e-3, 1000
WARMUP_STEPS = 500
CP_PATH_A, CP_PATH_B = 'checkpoint_A.pt', 'checkpoint_B.pt'
IGNORE_CHECKPOINT = True

import torch, os
model = GigaGraph_3B(vocab_size=VOCAB_SIZE, depth=DEPTH, d_model=D_MODEL)
resume_step = 0
if not IGNORE_CHECKPOINT:
    latest_cp = CP_PATH_A if os.path.exists(CP_PATH_A) else (CP_PATH_B if os.path.exists(CP_PATH_B) else None)
    if latest_cp: resume_step = model.load_checkpoint(latest_cp)

pipeline = GigaDataPipeline()
loader = pipeline.get_dataloader(batch_size=2, seq_len=1024, skip_steps=resume_step)
wandb.init(project='gigagraph-3b-cold-start', resume='allow', id='gigagraph-3b-run-1')

print('🚀 GigaGraph 3B Launching (Step: ' + str(resume_step) + ')...')
pbar = tqdm(loader)
for i, batch in enumerate(pbar):
    global_step = resume_step + i
    batch_cuda = batch.to('cuda:0')
    x, y = batch_cuda, torch.roll(batch_cuda, -1, dims=1)
    
    curr_lr = LEARNING_RATE * min(1.0, (global_step + 1) / WARMUP_STEPS)
    loss = model.train_step(x, y, lr=curr_lr)
    if global_step % 5 == 0:
        wandb.log({'loss': loss.item(), 'step': global_step}, commit=True)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{curr_lr:.2e}'})
        if loss > 200 or torch.isnan(loss):
            print('⚠️ EXPLOSION DETECTED. Terminating...')
            break
    if global_step > 0 and global_step % SAVE_INTERVAL == 0:
        cp_to_save = CP_PATH_A if (global_step // SAVE_INTERVAL) % 2 == 1 else CP_PATH_B
        model.save_checkpoint(cp_to_save, global_step)
wandb.finish()
